# Reproducible experiment — Multi-Objective HAPS Placement for Improved GNSS Positioning in Dense Urban Environments

This notebook is the executable scientific record for the ACDSA 2027 experiment. It evaluates four HAPS over **Shinjuku (Tokyo), Downtown Dubai, and Frankfurt Bankenviertel** using the published DESIGN/TEST protocol.

The computation includes the GPS/Galileo timing-group model, consistent ENU line-of-sight geometry, four-HAPS placement search, DESIGN-only model selection, held-out TEST evaluation, fixed-denominator quality curves, and durable checkpoints for long runs.

The experiment follows this sequence:

**verified inputs → GNSS geometry → DESIGN search → DESIGN layout selection → TEST evaluation → reproducibility evidence.**

TEST quantities never alter optimizer settings, seeds, search ranges, objectives, or selected layouts.


In [ ]:

from __future__ import annotations
import os, sys, json, math, time, hashlib, platform, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from scipy.stats import chi2, qmc
from IPython.display import display

PROTOCOL_VERSION = "ACDSA2027_PUBLIC_REPRODUCTION_V1"
os.environ.setdefault("MPLBACKEND","Agg")
SMOKE_TEST = os.environ.get("HAPS_GNSS_SMOKE_TEST", "0") == "1"
RUN_FULL = not SMOKE_TEST

PROJECT_ROOT = Path(os.environ.get("HAPS_GNSS_ROOT", Path.cwd())).resolve()
INPUTS = PROJECT_ROOT / "inputs"
CONFIG_PATH = Path(os.environ.get(
    "HAPS_GNSS_CONFIG",
    PROJECT_ROOT / "config" / "experiment.json"
)).resolve()

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing config: {CONFIG_PATH}")

CFG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
CONFIG_SHA256 = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
OUTPUT_ROOT = Path(os.environ.get("HAPS_GNSS_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).resolve()
# Stable full-run ID is intentional: resubmitting the same job resumes the same run.
DEFAULT_RUN_ID = ("smoke_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")) if SMOKE_TEST else "reproduction_run"
RUN_ID = os.environ.get("HAPS_GNSS_RUN_ID", DEFAULT_RUN_ID)
RUN_DIR = OUTPUT_ROOT / RUN_ID
RAW = RUN_DIR / "raw"
FIG = RUN_DIR / "figures"
CACHE = RUN_DIR / "cache"
for d in (RAW, FIG, CACHE):
    d.mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path) -> str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()


def canonical_sha256(obj) -> str:
    payload=json.dumps(obj,sort_keys=True,separators=(",",":"),default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def atomic_write_text(path: Path, text: str):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+".tmp")
    with tmp.open("w",encoding="utf-8") as f:
        f.write(text); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,path)


def atomic_write_json(path: Path, obj):
    atomic_write_text(path,json.dumps(obj,indent=2,default=str,allow_nan=True))


def atomic_write_df(path: Path, df: pd.DataFrame):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+".tmp")
    df.to_csv(tmp,index=False)
    with tmp.open("rb") as f:
        os.fsync(f.fileno())
    os.replace(tmp,path)


def append_jsonl_durable(path: Path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    line=json.dumps(obj,default=str,allow_nan=True,separators=(",",":"))
    with path.open("a",encoding="utf-8") as f:
        f.write(line+"\n"); f.flush(); os.fsync(f.fileno())


def load_jsonl_by_key(path: Path, key_field="key"):
    out={}
    path=Path(path)
    if not path.is_file(): return out
    with path.open("r",encoding="utf-8") as f:
        for line_no,line in enumerate(f,1):
            line=line.strip()
            if not line: continue
            try:
                rec=json.loads(line)
            except json.JSONDecodeError:
                # A crash can leave one incomplete final line. Ignore only malformed records.
                continue
            k=rec.get(key_field)
            if k is not None: out[str(k)]=rec
    return out

ROIS = CFG["rois"]
K_HAPS = int(CFG["placement"]["k_haps"])
MASK_DEG = float(CFG["dop"]["elevation_mask_deg"])
RX_H_M = float(CFG["grid"]["receiver_height_agl_m"])
GRID_SPACING_M = float(CFG["grid"]["receiver_spacing_m"])
HAPS_ALT_M = float(CFG["placement"]["altitude_m"])
HAPS_SIGMA_M = float(CFG["placement"]["sigma_m"])
PEB_CONF = float(CFG["crlb"]["peb_confidence"])
SIGMA_URE_DEFAULT_M = float(CFG["crlb"]["sigma_ure_default_m"])
MAX_RAYS_BATCH = int(CFG["raytracing"]["max_rays_per_batch"])
STEP_S = int(CFG["skyplot"]["time_step_s"])

def utc_naive(x):
    t=pd.Timestamp(x)
    if t.tzinfo is not None:
        t=t.tz_convert("UTC").tz_localize(None)
    return t

WIN=CFG["skyplot"]["time_window_utc"]
DESIGN_UTC=CFG["epochs"]["design_split"]["utc"]
TEST_UTC=CFG["epochs"]["held_out_split"]["utc"]
EPOCHS=list(pd.date_range(utc_naive(WIN[0]),utc_naive(WIN[1]),freq=f"{STEP_S}s"))
DESIGN_EPOCHS=list(pd.date_range(utc_naive(DESIGN_UTC[0]),utc_naive(DESIGN_UTC[1]),freq=f"{STEP_S}s"))
TEST_EPOCHS=list(pd.date_range(utc_naive(TEST_UTC[0]),utc_naive(TEST_UTC[1]),freq=f"{STEP_S}s"))

SEED_LABELS=list(CFG["algorithms"]["fair_comparison"]["seed_labels"])
N_EVAL=int(CFG["algorithms"]["fair_comparison"]["n_eval_budget"])
SEED_INTS={lbl:int("20260906"+str(lbl).split("_")[-1]) for lbl in SEED_LABELS}
R_MAX_M=(HAPS_ALT_M-RX_H_M)/math.tan(math.radians(MASK_DEG))
CHI2_95=float(chi2.ppf(PEB_CONF,df=2))

assert K_HAPS==4
assert len(EPOCHS)==25 and len(DESIGN_EPOCHS)==18 and len(TEST_EPOCHS)==7
assert N_EVAL==1000 and len(SEED_LABELS)==5

# Early resume guard: a previous full run may be resumed only with the same protocol and config.
if RUN_FULL:
    guard_path=RUN_DIR/"RESUME_GUARD.json"
    expected_guard={"protocol_version":PROTOCOL_VERSION,"config_sha256":CONFIG_SHA256}
    if guard_path.is_file():
        observed=json.loads(guard_path.read_text(encoding="utf-8"))
        if observed!=expected_guard:
            raise RuntimeError(
                "Resume guard mismatch. Do not mix outputs from different notebook/config versions. "
                "Use a new HAPS_GNSS_RUN_ID or remove the old run directory."
            )
    else:
        atomic_write_json(guard_path,expected_guard)

print(f"[MODE] {'SMOKE TEST' if SMOKE_TEST else 'FULL SCIENTIFIC RUN'}")
print(f"[RUN] {RUN_ID}")
print(f"[PROTOCOL] {PROTOCOL_VERSION}")
print(f"[ROIs] {[r['id'] for r in ROIS]}")


## 1. State model, numerical rank, and horizontal PEB95

The receiver state follows the visible time groups. A single visible time group uses `[e,n,u,b]`. When both GPS/HAPS-referenced measurements and Galileo measurements are visible, separate clock columns are used, equivalent to a common receiver clock plus a Galileo–GPS inter-system bias.

All spatial line-of-sight rows use one ENU convention, and positioning quality is the 95% horizontal PEB obtained from the weighted CRLB.


In [ ]:

def enu_unit_rows(az_deg, el_deg):
    az=np.radians(np.asarray(az_deg,float))
    el=np.radians(np.asarray(el_deg,float))
    return np.stack([
        np.sin(az)*np.cos(el),
        np.cos(az)*np.cos(el),
        np.sin(el)
    ],axis=-1)

def _svd_rank_batch(G, mask, n_state):
    Gm=np.where(mask[...,None],G,0.0)
    sv=np.linalg.svd(Gm,compute_uv=False)
    nlos=mask.sum(axis=1)
    smax=sv[:,0] if sv.shape[1] else np.zeros(G.shape[0],float)
    tol=np.maximum(n_state,nlos)*np.finfo(float).eps*smax
    rank=(sv>tol[:,None]).sum(axis=1).astype(np.int8)
    sreq=sv[:,n_state-1] if sv.shape[1]>=n_state else np.zeros(G.shape[0],float)
    margin=np.divide(sreq,smax,out=np.zeros_like(sreq),where=smax>0)
    return rank,margin

def _peb_batch(G, mask, sigma, valid):
    R,S,D=G.shape
    out=np.full(R,np.nan,float)
    if not np.any(valid):
        return out
    sig=np.broadcast_to(np.asarray(sigma,float),(R,S))
    sinel=np.clip(G[...,2],0.0,None)
    w=np.where(mask,sinel**2/(sig**2),0.0)
    A=np.einsum("rsi,rs,rsj->rij",G,w,G,optimize=True)
    for i in np.where(valid)[0]:
        try:
            C=np.linalg.inv(A[i])
            lam=float(np.max(np.linalg.eigvalsh(C[:2,:2])))
            if np.isfinite(lam) and lam>=0:
                out[i]=math.sqrt(CHI2_95*lam)
        except np.linalg.LinAlgError:
            pass
    return out

def solve_grouped_geometry(spatial, los, sigma, groups):
    """
    groups: 0 = GPS/HAPS time group, 1 = Galileo time group.
    If both groups are visible, use separate clock columns (5 states).
    If only one group is visible, use one clock column (4 states).
    """
    spatial=np.asarray(spatial,float)
    los=np.asarray(los,bool)
    groups=np.asarray(groups,int)
    R,S,_=spatial.shape
    if groups.shape!=(S,):
        raise ValueError("groups must have one entry per source")

    has0=np.any(los & (groups[None,:]==0),axis=1)
    has1=np.any(los & (groups[None,:]==1),axis=1)
    both=has0 & has1

    rank=np.zeros(R,np.int8)
    required=np.full(R,4,np.int8)
    margin=np.zeros(R,float)
    peb=np.full(R,np.nan,float)

    # One visible time group -> 4-state model.
    G4=np.concatenate([spatial,np.ones((R,S,1),float)],axis=2)
    r4,m4=_svd_rank_batch(G4,los,4)
    single=~both
    rank[single]=r4[single]
    margin[single]=m4[single]
    v4=single & (r4==4)
    p4=_peb_batch(G4,los,sigma,v4)
    peb[v4]=p4[v4]

    # Both time groups -> 5-state model with separate clock columns.
    c0=np.broadcast_to((groups==0).astype(float)[None,:,None],(R,S,1))
    c1=np.broadcast_to((groups==1).astype(float)[None,:,None],(R,S,1))
    G5=np.concatenate([spatial,c0,c1],axis=2)
    r5,m5=_svd_rank_batch(G5,los,5)
    required[both]=5
    rank[both]=r5[both]
    margin[both]=m5[both]
    v5=both & (r5==5)
    p5=_peb_batch(G5,los,sigma,v5)
    peb[v5]=p5[v5]

    valid=rank==required
    return {
        "rank":rank,"required_rank":required,"valid":valid,
        "peb95":peb,"sv_margin":margin,
        "has_base_group":has0,"has_galileo_group":has1
    }

def nondominated_rows(df, f1_col="design_f1", f2_col="design_f2"):
    d=df.copy().reset_index(drop=True)
    if len(d)==0:
        return d
    f1=pd.to_numeric(d[f1_col],errors="coerce").to_numpy(float)
    f2=pd.to_numeric(d[f2_col],errors="coerce").to_numpy(float)
    f2o=np.where(np.isfinite(f2),f2,np.inf)
    keep=np.ones(len(d),bool)
    for i in range(len(d)):
        dom=(f1>=f1[i])&(f2o<=f2o[i])&((f1>f1[i])|(f2o<f2o[i]))
        dom[i]=False
        if np.any(dom):
            keep[i]=False
    return d.loc[keep].copy()

def common_population_ecdf(peb_or_inf):
    x=np.asarray(peb_or_inf,float)
    n=len(x)
    finite=np.sort(x[np.isfinite(x)])
    if n==0 or len(finite)==0:
        return pd.DataFrame(columns=["peb95_threshold_m","fraction_all_gnss_unsolved"])
    return pd.DataFrame({
        "peb95_threshold_m":finite,
        "fraction_all_gnss_unsolved":np.arange(1,len(finite)+1,dtype=float)/n
    })

def select_design_extremes(df, group_cols):
    """Generic DESIGN-only max-R/min-PEB selector for genuinely multi-objective candidate sets."""
    rows=[]
    for keys,g in df.groupby(group_cols,dropna=False):
        g=g.copy()
        maxr=g.sort_values(["design_f1","design_f2","placement_id"],
                           ascending=[False,True,True],na_position="last").iloc[0].copy()
        maxr["selection_rule"]="design_max_R"
        rows.append(maxr)
        finite=g[np.isfinite(pd.to_numeric(g["design_f2"],errors="coerce"))].copy()
        if len(finite):
            minp=finite.sort_values(["design_f2","design_f1","placement_id"],
                                    ascending=[True,False,True]).iloc[0].copy()
            minp["selection_rule"]="design_min_PEB"
            rows.append(minp)
    return pd.DataFrame(rows).drop_duplicates(
        subset=list(group_cols)+["placement_id","selection_rule"]
    ).reset_index(drop=True)


def select_paper_representatives(frozen_df):
    """
    DESIGN-only reporting rule.
    Greedy and DE optimize R only, so each gets only a recovery-selected representative.
    NSGA-II and structured multi-objective candidate sets get DESIGN max-R and DESIGN min-PEB extrema.
    """
    rows=[]
    for rid,gcity in frozen_df.groupby("roi_id"):
        # Greedy: the single completed four-HAPS layout.
        gg=gcity[gcity.algorithm=="greedy_add"]
        if len(gg):
            r=gg.sort_values(["design_f1","placement_id"],ascending=[False,True]).iloc[0].copy()
            r["selection_rule"]="design_R_solution"; rows.append(r.to_dict())
        # DE: choose the best seed using DESIGN recovery only; PEB is an associated outcome.
        gd=gcity[gcity.algorithm=="differential_evolution"]
        if len(gd):
            r=gd.sort_values(["design_f1","seed","placement_id"],
                             ascending=[False,True,True],na_position="last").iloc[0].copy()
            r["selection_rule"]="design_max_R"; rows.append(r.to_dict())
        # Multi-objective candidate sets.
        for algo in ["nsga2","symmetric_ring","center_plus_three"]:
            ga=gcity[gcity.algorithm==algo]
            if len(ga):
                rows.extend(select_design_extremes(ga,["roi_id","algorithm"]).to_dict("records"))
    return pd.DataFrame(rows).reset_index(drop=True)


def select_seed_representatives(frozen_df):
    """Per-seed DESIGN representatives used only for stochastic variability."""
    rows=[]
    de=frozen_df[frozen_df.algorithm=="differential_evolution"]
    for _,r in de.iterrows():
        z=r.copy(); z["selection_rule"]="design_R_solution"; rows.append(z.to_dict())
    ns=frozen_df[frozen_df.algorithm=="nsga2"]
    if len(ns):
        rows.extend(select_design_extremes(ns,["roi_id","algorithm","seed"]).to_dict("records"))
    return pd.DataFrame(rows).reset_index(drop=True)


In [ ]:

# Run numerical and checkpoint checks in both execution modes.
# These checks are independent of the reported experiment.

# ENU convention: az=0° -> North; az=90° -> East.
u=enu_unit_rows(np.array([0.0,90.0]),np.array([0.0,0.0]))
assert np.allclose(u[0],[0,1,0],atol=1e-12)
assert np.allclose(u[1],[1,0,0],atol=1e-12)

# 4-state single-time-group rank.
sp=np.array([[[1.0,0.0,0.4],[0.0,1.0,0.5],[-0.7,0.3,0.6],[0.4,-0.8,0.7]]])
sol4=solve_grouped_geometry(sp,np.ones((1,4),bool),np.ones(4),np.zeros(4,int))
assert sol4["required_rank"][0]==4 and sol4["rank"][0]==4

# 5-state two-time-group rank.
sp5=np.array([[[1.0,0.0,0.4],[0.0,1.0,0.5],[-0.7,0.3,0.6],[0.4,-0.8,0.7],[0.2,0.5,0.8]]])
sol5=solve_grouped_geometry(sp5,np.ones((1,5),bool),np.ones(5),np.array([0,0,0,0,1]))
assert sol5["required_rank"][0]==5 and sol5["rank"][0]==5

# Common-population curve denominator is fixed, including unsolved cases as +inf.
ec=common_population_ecdf([50.0,100.0,np.inf,np.inf])
assert np.allclose(ec["fraction_all_gnss_unsolved"].to_numpy(),[0.25,0.50])

# DESIGN-only reporting rules: DE never gets a synthetic "min-PEB" winner.
toy=pd.DataFrame([
    {"roi_id":"x","algorithm":"differential_evolution","seed":"s1","placement_id":"d1","design_f1":0.20,"design_f2":100.0,"test_f1":0.1},
    {"roi_id":"x","algorithm":"differential_evolution","seed":"s2","placement_id":"d2","design_f1":0.15,"design_f2":20.0,"test_f1":0.9},
    {"roi_id":"x","algorithm":"nsga2","seed":"s1","placement_id":"n1","design_f1":0.22,"design_f2":120.0,"test_f1":0.1},
    {"roi_id":"x","algorithm":"nsga2","seed":"s1","placement_id":"n2","design_f1":0.18,"design_f2":40.0,"test_f1":0.9},
])
reps=select_paper_representatives(toy)
de_reps=reps[reps.algorithm=="differential_evolution"]
assert len(de_reps)==1 and de_reps.iloc[0].placement_id=="d1" and de_reps.iloc[0].selection_rule=="design_max_R"
ns_reps=reps[reps.algorithm=="nsga2"]
assert set(ns_reps.selection_rule)=={"design_max_R","design_min_PEB"}
assert set(ns_reps.placement_id)=={"n1","n2"}

# Durable JSONL cache loader tolerates an incomplete final line.
smoke_cache=RUN_DIR/"_smoke_cache.jsonl"
append_jsonl_durable(smoke_cache,{"key":"a","value":1})
with smoke_cache.open("a",encoding="utf-8") as f: f.write('{"key":"broken"')
loaded=load_jsonl_by_key(smoke_cache)
assert loaded["a"]["value"]==1 and "broken" not in loaded

# Atomic CSV/JSON checkpoint helpers.
atomic_df=RUN_DIR/"_smoke_atomic.csv"
atomic_json=RUN_DIR/"_smoke_atomic.json"
atomic_write_df(atomic_df,pd.DataFrame({"x":[1,2]}))
atomic_write_json(atomic_json,{"ok":True})
assert pd.read_csv(atomic_df)["x"].tolist()==[1,2]
assert json.loads(atomic_json.read_text(encoding="utf-8"))["ok"] is True

print("[OK] core scientific and resume smoke tests passed")


## 2. Input verification and urban scenes

The three OpenStreetMap footprint datasets are used as fixed experimental inputs and verified by hash before execution. Building heights use tagged `height`/`building:height`; otherwise `building:levels × 3 m`; footprints without usable height information are excluded from the 3-D obstruction mesh.

The run also verifies the scientific source package and GNSS input hashes before any expensive computation. Resume checkpoints are accepted only when the complete input fingerprint matches.


In [ ]:

if RUN_FULL:
    import geopandas as gpd
    import trimesh
    import pymap3d as pm
    from src.haps_gnss import ephemeris as eph
    from src.haps_gnss import receiver_eval as reval
    from src.haps_gnss import reference_layout as layout_core
    from src.haps_gnss import osm_geometry as osmgeo

    # Verify the scientific modules used for propagation, LOS, and urban geometry.
    for rel,expected in CFG.get("source_hashes",{}).items():
        p=PROJECT_ROOT/rel
        if not p.is_file():
            raise FileNotFoundError(f"Missing authoritative source: {p}")
        obs=sha256_file(p)
        if obs.lower()!=expected.lower():
            raise RuntimeError(f"Source hash mismatch: {rel}\nexpected {expected}\nobserved {obs}")

    # Use the GNSS inputs specified for this experiment.
    BRDC_PATH=INPUTS/"brdc"/"BRDC_normalized_GPS_GAL.rnx"
    SV_PATH=INPUTS/"ephemeris"/"sv_table.csv"
    for p,key in [(BRDC_PATH,"BRDC_normalized_GPS_GAL.rnx"),(SV_PATH,"sv_table.csv")]:
        if not p.is_file():
            raise FileNotFoundError(p)
        expected=CFG["input_hashes"][key]
        obs=sha256_file(p)
        if obs.lower()!=expected.lower():
            raise RuntimeError(f"GNSS input hash mismatch: {p.name}")

    # Verify building-footprint data against recorded checksums.
    manifest_path=INPUTS/"CAMERA_READY_INPUT_MANIFEST.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"{manifest_path} is missing. Run prepare_new_city_osm.py before the scientific notebook."
        )
    input_manifest=json.loads(manifest_path.read_text(encoding="utf-8"))

    input_fingerprint={
        "protocol_version":PROTOCOL_VERSION,
        "config_sha256":CONFIG_SHA256,
        "source_hashes_observed":{},
        "gnss_input_hashes_observed":{},
        "footprint_hashes_observed":{},
        "input_manifest_sha256":sha256_file(manifest_path),
    }
    for rel in CFG.get("source_hashes",{}):
        input_fingerprint["source_hashes_observed"][rel]=sha256_file(PROJECT_ROOT/rel)
    input_fingerprint["gnss_input_hashes_observed"]["BRDC_normalized_GPS_GAL.rnx"]=sha256_file(BRDC_PATH)
    input_fingerprint["gnss_input_hashes_observed"]["sv_table.csv"]=sha256_file(SV_PATH)
    for roi in ROIS:
        gpkg=INPUTS/roi["id"]/"footprints.gpkg"
        input_fingerprint["footprint_hashes_observed"][roi["id"]]=sha256_file(gpkg)
    input_fingerprint["fingerprint_sha256"]=canonical_sha256(input_fingerprint)
    fingerprint_path=RUN_DIR/"INPUT_FINGERPRINT.json"
    if fingerprint_path.is_file():
        previous=json.loads(fingerprint_path.read_text(encoding="utf-8"))
        if previous!=input_fingerprint:
            raise RuntimeError("Input fingerprint mismatch: refusing to mix old checkpoints with changed scientific inputs.")
    else:
        atomic_write_json(fingerprint_path,input_fingerprint)
    INPUT_FINGERPRINT_SHA256=input_fingerprint["fingerprint_sha256"]

    SCENES={}
    geometry_rows=[]
    for roi in ROIS:
        rid=roi["id"]
        gpkg=INPUTS/rid/"footprints.gpkg"
        if not gpkg.is_file():
            raise FileNotFoundError(gpkg)
        rec_manifest=input_manifest["cities"][rid]
        obs=sha256_file(gpkg)
        if obs.lower()!=rec_manifest["sha256"].lower():
            raise RuntimeError(f"{rid}: footprints hash mismatch")

        frame=osmgeo.LocalFrame(*osmgeo.roi_center(roi))
        gdf=gpd.read_file(gpkg).reset_index(drop=True)
        stats=osmgeo.height_imputation_rate(gdf)
        mesh,excl=osmgeo.footprints_to_mesh(gdf,frame)
        scene=reval.build_scene(mesh)
        recs=reval.receiver_grid(roi,CFG["grid"],frame).copy()
        recs["rec_id"]=np.arange(len(recs),dtype=int)

        if len(recs)==0:
            raise RuntimeError(f"{rid}: empty receiver grid")
        SCENES[rid]={
            "roi":roi,"frame":frame,"scene":scene,"mesh":mesh,"recs":recs,
            "mesh_stats":{**stats,**excl}
        }
        geometry_rows.append({
            "roi_id":rid,"epsg":frame.crs.to_epsg(),"n_receivers":len(recs),
            **stats,**excl
        })

    geometry_df=pd.DataFrame(geometry_rows)
    atomic_write_df(RAW/"urban_geometry_summary.csv",geometry_df)
    display(geometry_df)
else:
    SCENES={r["id"]:None for r in ROIS}


## 3. GPS/Galileo geometry cache

Satellite propagation and geometric line-of-sight evaluation use the scientific modules included in this repository. The measurement-state construction follows the timing-group model defined above, and the spatial ENU rows use the same convention throughout.


In [ ]:

if RUN_FULL:
    SVDF=pd.read_csv(SV_PATH)
    SVDF["toc"]=pd.to_datetime(SVDF["toc"],utc=True).dt.tz_convert(None)
    TIME_OFFSETS=reval.time_system_offsets(BRDC_PATH)

    def build_gnss_cache(rid,epoch):
        cdir=CACHE/rid/"gnss_corrected"
        cdir.mkdir(parents=True,exist_ok=True)
        path=cdir/f"{pd.Timestamp(epoch).strftime('%Y%m%dT%H%M%S')}.npz"
        if path.exists():
            try:
                d=np.load(path,allow_pickle=False)
                required={"sv","azm","elm","los","sigma0","spatial","groups","rank","required_rank","p1","peb95","sv_margin","n_los"}
                if required.issubset(set(d.files)):
                    return path
            except Exception:
                pass
            path.unlink(missing_ok=True)
        s=SCENES[rid]
        sel=eph.select_epoch(SVDF,pd.Timestamp(epoch)).copy()
        sel["sigma_ure_m"]=[
            reval.sv_sigma_ure(row,SIGMA_URE_DEFAULT_M) for _,row in sel.iterrows()
        ]
        sv_geom=reval.propagate_svs(sel,pd.Timestamp(epoch),TIME_OFFSETS)
        prep=reval.prepare_epoch(pd.Timestamp(epoch),sv_geom,s["recs"],s["scene"],CFG)
        azm=np.asarray(prep["azm"],float)
        elm=np.asarray(prep["elm"],float)
        los=np.asarray(prep["los"],bool)
        sigma0=np.asarray(prep["sigv"],float)
        spatial=enu_unit_rows(azm,elm)
        sv=np.asarray(prep["sv_names"]).astype(str)
        groups=np.array([1 if x.startswith("E") else 0 for x in sv],dtype=np.int8)
        sol=solve_grouped_geometry(spatial,los,sigma0,groups)
        tmp=path.with_name(path.name+".tmp.npz")
        np.savez_compressed(
            tmp,epoch=str(epoch),sv=sv,azm=azm,los=los,elm=elm,
            sigma0=sigma0,spatial=spatial,groups=groups,
            rank=sol["rank"],required_rank=sol["required_rank"],
            p1=sol["valid"],peb95=sol["peb95"],sv_margin=sol["sv_margin"],
            n_los=los.sum(axis=1)
        )
        os.replace(tmp,path)
        return path

    print("[RUN] building/loading corrected GNSS caches")
    for rid in SCENES:
        for ep in EPOCHS:
            build_gnss_cache(rid,ep)
        print(f"[OK] {rid}: {len(EPOCHS)} epochs")

    # Summarize DESIGN solvability using the GPS/Galileo clock-state model.
    design_population=[]
    for rid in SCENES:
        n_total=0;n_un=0
        for ep in DESIGN_EPOCHS:
            d=np.load(build_gnss_cache(rid,ep),allow_pickle=False)
            n_total+=len(d["p1"]); n_un+=int((~d["p1"]).sum())
        design_population.append({
            "roi_id":rid,"n_receiver_epochs":n_total,
            "n_gnss_unsolved":n_un,
            "fraction_gnss_unsolved":n_un/n_total
        })
    design_population_df=pd.DataFrame(design_population)
    atomic_write_df(RAW/"design_population_summary.csv",design_population_df)
    display(design_population_df)


## 4. HAPS geometry and objective evaluation

Four static HAPS are placed at 20 km altitude. Each platform is controlled by a horizontal polar angle and radius. The polar angle is measured from local East (`e=r cos(theta)`, `n=r sin(theta)`), while LOS azimuth follows the navigation convention measured clockwise from North. HAPS observations share the synchronized GPS/common-time group; Galileo uses the second timing group when both are visible.

Every DESIGN or TEST placement evaluation is written to a durable cache so interrupted long runs can resume without changing the scientific sequence.


In [ ]:

if RUN_FULL:
    def placement_key(thetas,rs):
        vals=np.concatenate([np.asarray(thetas,float),np.asarray(rs,float)])
        return hashlib.sha256("|".join(f"{v:.17g}" for v in vals).encode()).hexdigest()

    def placement_haps(rid,thetas,rs):
        roi=SCENES[rid]["roi"]
        lon0,lat0=osmgeo.roi_center(roi)
        th=np.asarray(thetas,float); rr=np.asarray(rs,float)
        east=rr*np.cos(th); north=rr*np.sin(th)
        lat,lon,_=pm.enu2geodetic(east,north,np.zeros(len(rr)),lat0,lon0,0.0,deg=True)
        return np.column_stack([np.asarray(lon,float),np.asarray(lat,float),
                                np.full(len(rr),HAPS_ALT_M,float)])

    def haps_observations(rid,haps_lonlatalt):
        s=SCENES[rid]; recs=s["recs"]
        prep_stub={
            "recs":recs,
            "mask_deg":MASK_DEG,
            "origins":np.column_stack([
                recs["x_m"].to_numpy(float),recs["y_m"].to_numpy(float),
                np.full(len(recs),RX_H_M,float)
            ]),
            "max_batch":MAX_RAYS_BATCH
        }
        haz,hel,hrng,hlos=layout_core._haps_los(prep_stub,s["scene"],CFG,haps_lonlatalt)
        return {
            "azm":np.asarray(haz,float),"elm":np.asarray(hel,float),
            "los":np.asarray(hlos,bool),
            "spatial":enu_unit_rows(haz,hel),
            "sigma":np.full(haps_lonlatalt.shape[0],HAPS_SIGMA_M,float),
            "groups":np.zeros(haps_lonlatalt.shape[0],dtype=np.int8)
        }

    def score_epoch_with_haps(rid,epoch,hobs):
        d=np.load(build_gnss_cache(rid,epoch),allow_pickle=False)
        spatial=np.concatenate([d["spatial"],hobs["spatial"]],axis=1)
        los=np.concatenate([d["los"],hobs["los"]],axis=1)
        sigma=np.concatenate([d["sigma0"],hobs["sigma"]])
        groups=np.concatenate([d["groups"],hobs["groups"]])
        sol=solve_grouped_geometry(spatial,los,sigma,groups)
        p1=d["p1"].astype(bool)
        p2=(~p1)&sol["valid"]
        peb=sol["peb95"]
        return {
            "n_unpos":int((~p1).sum()),"n_p2":int(p2.sum()),
            "p1":p1,"p2":p2,"peb95_all":peb,
            "rank_aug":sol["rank"],"required_rank_aug":sol["required_rank"],
            "sv_margin_aug":sol["sv_margin"],
            "n_los_gnss":d["n_los"],"n_los_aug":los.sum(axis=1)
        }

    def aggregate_epoch_scores(results):
        n_un=sum(r["n_unpos"] for r in results)
        n_p2=sum(r["n_p2"] for r in results)
        f1=n_p2/n_un if n_un else np.nan
        vals=[]
        for r in results:
            vals.extend(r["peb95_all"][r["p2"]].tolist())
        vals=np.asarray(vals,float)
        nonfinite=int((~np.isfinite(vals)).sum()) if len(vals) else 0
        quality_ok=(n_p2>0 and len(vals)==n_p2 and nonfinite==0)
        med=float(np.median(vals)) if quality_ok else np.nan
        return {
            "f1":float(f1),"f2":med,
            "f2_order":med if quality_ok else float("inf"),
            "n_unpos":n_un,"n_p2":n_p2,
            "peb95_median":med,
            "peb95_p95":float(np.percentile(vals,95)) if quality_ok else np.nan,
            "peb95_max":float(np.max(vals)) if quality_ok else np.nan,
            "n_nonfinite_peb":nonfinite
        }

    def eval_cache_path(rid,split):
        return CACHE/rid/"placement_eval"/f"{split}.jsonl"

    EVAL_CACHE={rid:{
        "design":load_jsonl_by_key(eval_cache_path(rid,"design")),
        "test":load_jsonl_by_key(eval_cache_path(rid,"test"))
    } for rid in SCENES}
    for rid in SCENES:
        print(f"[RESUME] {rid}: {len(EVAL_CACHE[rid]['design'])} DESIGN + {len(EVAL_CACHE[rid]['test'])} TEST placement evaluations cached")

    def score_placement(rid,thetas,rs,split="design",detail=False):
        if split not in ("design","test"):
            raise ValueError(split)
        key=placement_key(thetas,rs)
        if not detail and key in EVAL_CACHE[rid][split]:
            return EVAL_CACHE[rid][split][key]
        hp=placement_haps(rid,thetas,rs)
        hobs=haps_observations(rid,hp)
        eps=DESIGN_EPOCHS if split=="design" else TEST_EPOCHS
        results=[score_epoch_with_haps(rid,ep,hobs) for ep in eps]
        agg=aggregate_epoch_scores(results)
        out={"key":key,"thetas":[float(x) for x in thetas],
             "rs":[float(x) for x in rs],**agg}
        if detail:
            rows=[]
            for ep,r in zip(eps,results):
                ids=np.where(~r["p1"])[0]
                if len(ids):
                    rows.append(pd.DataFrame({
                        "epoch":str(ep),"rec_id":ids,
                        "made_solvable":r["p2"][ids],
                        "peb95_m":np.where(r["p2"][ids],r["peb95_all"][ids],np.inf),
                        "rank_aug":r["rank_aug"][ids],
                        "required_rank_aug":r["required_rank_aug"][ids],
                        "n_los_gnss":r["n_los_gnss"][ids],
                        "n_los_aug":r["n_los_aug"][ids],
                    }))
            out["_detail"]=pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()
        else:
            cache_record={**out,"protocol_version":PROTOCOL_VERSION,
                          "input_fingerprint_sha256":INPUT_FINGERPRINT_SHA256}
            append_jsonl_durable(eval_cache_path(rid,split),cache_record)
            EVAL_CACHE[rid][split][key]=out
        return out

    def score_fixed_reference(rid,split="design",detail=False):
        eps=DESIGN_EPOCHS if split=="design" else TEST_EPOCHS
        results=[]
        details=[]
        roi=SCENES[rid]["roi"]
        center_latlon=(osmgeo.roi_center(roi)[1],osmgeo.roi_center(roi)[0])
        for ep in eps:
            hp=layout_core.haps_ring_k(pd.Timestamp(ep),CFG["haps_overlay"],center_latlon,4)
            hobs=haps_observations(rid,hp)
            r=score_epoch_with_haps(rid,ep,hobs)
            results.append(r)
            if detail:
                ids=np.where(~r["p1"])[0]
                if len(ids):
                    details.append(pd.DataFrame({
                        "epoch":str(ep),"rec_id":ids,
                        "made_solvable":r["p2"][ids],
                        "peb95_m":np.where(r["p2"][ids],r["peb95_all"][ids],np.inf),
                        "rank_aug":r["rank_aug"][ids],
                        "required_rank_aug":r["required_rank_aug"][ids],
                        "n_los_gnss":r["n_los_gnss"][ids],
                        "n_los_aug":r["n_los_aug"][ids],
                    }))
        out=aggregate_epoch_scores(results)
        if detail:
            out["_detail"]=pd.concat(details,ignore_index=True) if details else pd.DataFrame()
        return out

    print(f"[OK] placement radius: 0–{R_MAX_M:.3f} m")


## 5. Unrestricted DESIGN optimization — Greedy, DE and NSGA-II

The search budgets and seeds are specified in the experiment configuration. No TEST epoch is accessed in this section.

Resume behavior is conservative: completed algorithm/seed outputs are loaded directly. If interruption occurs inside an unfinished DE/NSGA-II seed, the same deterministic seed is restarted, but the disk-backed placement cache makes already completed scientific objective evaluations reusable instead of recomputing ray tracing and positioning metrics.


In [ ]:
if RUN_FULL:
    # ============================================================
    # Section 10 — Run Greedy-Add baseline (step-resumable)
    # ============================================================
    GREEDY = {}
    theta_grid = np.arange(24,dtype=float)*(2*np.pi/24)
    r_grid = np.arange(11,dtype=float)*(R_MAX_M/10)
    logical_evals_per_step=len(theta_grid)*len(r_grid)

    for rid in SCENES:
        ck=RAW/rid/"greedy_steps.csv"
        ck.parent.mkdir(parents=True,exist_ok=True)
        if ck.exists():
            steps=pd.read_csv(ck)
            if len(steps)>K_HAPS or (len(steps) and steps["step"].astype(int).tolist()!=list(range(1,len(steps)+1))):
                raise RuntimeError(f"{rid}: malformed Greedy checkpoint {ck}")
            placed_t=steps["theta"].astype(float).tolist()
            placed_r=steps["r_m"].astype(float).tolist()
            step_rows=steps.to_dict("records")
            print(f"[RESUME] {rid}: Greedy completed steps={len(step_rows)}/4")
        else:
            placed_t=[]; placed_r=[]; step_rows=[]

        for step in range(len(step_rows)+1,K_HAPS+1):
            candidates=[]
            for th in theta_grid:
                for rr in r_grid:
                    res=score_placement(rid,placed_t+[float(th)],placed_r+[float(rr)],"design")
                    candidates.append((res["f1"], float(th), float(rr), res))
            max_f1=max(x[0] for x in candidates)
            tied=[x for x in candidates if x[0] == max_f1]
            _,th,rr,res=min(tied,key=lambda x:(x[1],x[2]))
            placed_t.append(th); placed_r.append(rr)
            step_rows.append({
                "step":step,"theta":th,"theta_deg":np.degrees(th),"r_m":rr,
                "f1_after":res["f1"],"f2_median_peb95_after":res["f2"],
                "n_eval_cumulative":step*logical_evals_per_step
            })
            # One durable checkpoint after every completed Greedy stage.
            atomic_write_df(ck,pd.DataFrame(step_rows))
            print(f"[RUN] {rid} Greedy step {step}/4 | f1={res['f1']:.6f} | theta={np.degrees(th):.1f}° | r={rr:.1f} m")

        steps=pd.DataFrame(step_rows)
        assert len(steps)==4 and int(steps.iloc[-1].n_eval_cumulative)==1056
        GREEDY[rid]={"thetas":placed_t,"rs":placed_r,"steps":steps,"n_eval":1056}

    greedy_df=pd.concat([g["steps"].assign(roi_id=rid) for rid,g in GREEDY.items()],ignore_index=True)
    display(greedy_df)

    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(7,4))
    for rid,g in GREEDY.items():
        ax.plot(g["steps"]["step"],g["steps"]["f1_after"],marker="o",label=rid)
    ax.set_xlabel("Greedy HAPS count"); ax.set_ylabel("DESIGN recovery f1")
    ax.set_xticks([1,2,3,4]); ax.legend()
    fig.tight_layout(); fig.savefig(FIG/"greedy_progression.png",dpi=180); plt.close(fig)


In [ ]:
if RUN_FULL:
    # ============================================================
    # Section 11 — Run Differential Evolution
    # ------------------------------------------------------------
    # Purpose:
    # - Run Differential Evolution using the specified settings.
    # ============================================================

    DE_RESULTS={}
    BOUNDS=np.array([[0,2*np.pi],[0,R_MAX_M]]*K_HAPS,float)

    def decode_x(x):
        x=np.asarray(x,float)
        return x[0::2].tolist(),x[1::2].tolist()

    for rid in SCENES:
        DE_RESULTS[rid]={}
        outdir=RAW/rid/"de"; outdir.mkdir(parents=True,exist_ok=True)
        for lbl in SEED_LABELS:
            ck=outdir/f"{lbl}.json"
            curve_path=outdir/f"{lbl}_curve.csv"
            if ck.exists() and curve_path.exists():
                meta=json.loads(ck.read_text(encoding="utf-8"))
                curve=pd.read_csv(curve_path)
                DE_RESULTS[rid][lbl]={"meta":meta,"curve":curve}
                print(f"[SKIP] {rid} DE {lbl}: checkpoint loaded")
                continue

            seed=SEED_INTS[lbl]
            sampler=qmc.LatinHypercube(d=2*K_HAPS,seed=seed)
            init=qmc.scale(sampler.random(100),BOUNDS[:,0],BOUNDS[:,1])
            evals=[]
            def objective(x):
                th,rr=decode_x(x)
                rec=score_placement(rid,th,rr,"design")
                evals.append({"x":np.asarray(x,float).tolist(),
                              "f1":float(rec["f1"]),
                              "f2":float(rec["f2_order"])})
                return -float(rec["f1"])

            t0=time.perf_counter()
            differential_evolution(
                objective,
                bounds=[tuple(b) for b in BOUNDS],
                init=init,
                strategy="best1bin",
                maxiter=9,
                mutation=(0.5,1.0),
                recombination=0.7,
                polish=False,
                updating="immediate",
                workers=1,
                seed=seed,
                tol=0.0,
                atol=0.0,
                disp=False,
            )
            if len(evals) != N_EVAL:
                raise RuntimeError(f"{rid} DE {lbl}: {len(evals)} evals != frozen {N_EVAL}")

            best_idx=max(range(len(evals)),key=lambda i:evals[i]["f1"])
            best=evals[best_idx]
            curve=[]
            best_sofar=-np.inf
            for i,e in enumerate(evals,1):
                best_sofar=max(best_sofar,e["f1"])
                row={"eval_idx":i,"best_f1_sofar":best_sofar,"f1":e["f1"],"f2":e["f2"]}
                for j in range(K_HAPS):
                    row[f"x_theta{j+1}"]=e["x"][2*j]
                    row[f"x_r{j+1}"]=e["x"][2*j+1]
                curve.append(row)
            curve=pd.DataFrame(curve)
            atomic_write_df(curve_path,curve)
            meta={"seed_label":lbl,"seed_int":seed,"n_eval_used":len(evals),
                  "best_idx":best_idx,"best_x":best["x"],"best_f1":best["f1"],
                  "best_f2":best["f2"],"status":"budget_complete",
                  "wall_s":time.perf_counter()-t0}
            atomic_write_json(ck,meta)
            DE_RESULTS[rid][lbl]={"meta":meta,"curve":curve}
            print(f"[OK] {rid} DE {lbl}: 1000 evals | best DESIGN f1={best['f1']:.6f}")

    de_summary=pd.DataFrame([
        {"roi_id":rid,"seed":lbl,**{k:v for k,v in d["meta"].items()
                                    if k in ("n_eval_used","best_f1","best_f2","status","wall_s")}}
        for rid,rr in DE_RESULTS.items() for lbl,d in rr.items()
    ])
    display(de_summary)

    for rid,rr in DE_RESULTS.items():
        fig,ax=plt.subplots(figsize=(7,4))
        for lbl,d in rr.items():
            ax.plot(d["curve"]["eval_idx"],d["curve"]["best_f1_sofar"],label=lbl)
        ax.set_xlabel("Objective evaluations"); ax.set_ylabel("Best DESIGN recovery f1")
        ax.set_title(f"DE convergence — {rid}"); ax.legend(fontsize=8)
        fig.tight_layout(); fig.savefig(FIG/f"de_convergence_{rid}.png",dpi=180); plt.show()


In [ ]:
if RUN_FULL:
    # ============================================================
    # Section 12 — Run NSGA-II
    # ------------------------------------------------------------
    # Purpose:
    # - Run NSGA-II using the specified settings.
    # ============================================================

    try:
        from pymoo.algorithms.moo.nsga2 import NSGA2
        from pymoo.core.problem import Problem
        from pymoo.optimize import minimize
        from pymoo.factory import get_sampling, get_crossover, get_mutation
    except Exception as exc:
        raise ImportError("pymoo==0.5.0 is required for the frozen NSGA-II method") from exc

    def nondominated_indices(evals):
        f1=np.array([e["f1"] for e in evals],float)
        f2=np.array([e["f2"] for e in evals],float)
        out=[]
        for i in range(len(evals)):
            dominated=np.any(
                (f1 >= f1[i]) & (f2 <= f2[i]) &
                ((f1 > f1[i]) | (f2 < f2[i]))
            )
            if not dominated:
                out.append(i)
        return out

    NSGA_RESULTS={}
    for rid in SCENES:
        NSGA_RESULTS[rid]={}
        outdir=RAW/rid/"nsga2"; outdir.mkdir(parents=True,exist_ok=True)
        for lbl in SEED_LABELS:
            ck=outdir/f"{lbl}.json"
            curve_path=outdir/f"{lbl}_curve.csv"
            front_path=outdir/f"{lbl}_front.csv"
            if ck.exists() and curve_path.exists() and front_path.exists():
                meta=json.loads(ck.read_text(encoding="utf-8"))
                curve=pd.read_csv(curve_path); front=pd.read_csv(front_path)
                NSGA_RESULTS[rid][lbl]={"meta":meta,"curve":curve,"front":front}
                print(f"[SKIP] {rid} NSGA-II {lbl}: checkpoint loaded")
                continue

            seed=SEED_INTS[lbl]
            evals=[]
            class PlacementProblem(Problem):
                def __init__(self):
                    super().__init__(
                        n_var=2*K_HAPS,n_obj=2,n_constr=0,
                        xl=BOUNDS[:,0],xu=BOUNDS[:,1]
                    )
                def _evaluate(self,X,out,*args,**kwargs):
                    F=[]
                    for x in np.atleast_2d(X):
                        th,rr=decode_x(x)
                        rec=score_placement(rid,th,rr,"design")
                        f1=float(rec["f1"]); f2=float(rec["f2_order"])
                        evals.append({"x":np.asarray(x,float).tolist(),"f1":f1,"f2":f2})
                        F.append([-f1, f2 if np.isfinite(f2) else 1e30])
                    out["F"]=np.asarray(F,float)

            algorithm=NSGA2(
                pop_size=100,
                sampling=get_sampling("real_lhs"),
                crossover=get_crossover("real_sbx",prob=0.9,eta=15),
                mutation=get_mutation("real_pm",eta=20),
                eliminate_duplicates=False
            )
            t0=time.perf_counter()
            minimize(PlacementProblem(),algorithm,("n_eval",N_EVAL),
                     seed=seed,verbose=False,save_history=False)
            if len(evals) != N_EVAL:
                raise RuntimeError(f"{rid} NSGA-II {lbl}: {len(evals)} evals != frozen {N_EVAL}")

            front_idx=nondominated_indices(evals)
            best_sofar=-np.inf; curve=[]
            for i,e in enumerate(evals,1):
                best_sofar=max(best_sofar,e["f1"])
                row={"eval_idx":i,"best_f1_sofar":best_sofar,"f1":e["f1"],"f2":e["f2"]}
                for j in range(K_HAPS):
                    row[f"x_theta{j+1}"]=e["x"][2*j]
                    row[f"x_r{j+1}"]=e["x"][2*j+1]
                curve.append(row)
            curve=pd.DataFrame(curve); atomic_write_df(curve_path,curve)

            front_rows=[]
            for fi in front_idx:
                e=evals[fi]
                row={"eval_idx":fi+1,"f1":e["f1"],"f2":e["f2"]}
                for j in range(K_HAPS):
                    row[f"x_theta{j+1}"]=e["x"][2*j]
                    row[f"x_r{j+1}"]=e["x"][2*j+1]
                front_rows.append(row)
            front=pd.DataFrame(front_rows)
            atomic_write_df(front_path,front)
            meta={"seed_label":lbl,"seed_int":seed,"n_eval_used":len(evals),
                  "n_front_points":len(front_idx),"status":"budget_complete",
                  "wall_s":time.perf_counter()-t0}
            atomic_write_json(ck,meta)
            NSGA_RESULTS[rid][lbl]={"meta":meta,"curve":curve,"front":front}
            print(f"[OK] {rid} NSGA-II {lbl}: 1000 evals | front={len(front_idx)} points")

    nsga_summary=pd.DataFrame([
        {"roi_id":rid,"seed":lbl,**d["meta"]}
        for rid,rr in NSGA_RESULTS.items() for lbl,d in rr.items()
    ])
    display(nsga_summary[["roi_id","seed","n_eval_used","n_front_points","status"]])

    for rid,rr in NSGA_RESULTS.items():
        fig,ax=plt.subplots(figsize=(6,4))
        for lbl,d in rr.items():
            f=d["front"]
            if len(f):
                ax.scatter(f["f2"],f["f1"],s=18,label=lbl)
        ax.set_xlabel("DESIGN median recovered PEB95 (m)")
        ax.set_ylabel("DESIGN recovery f1")
        ax.set_title(f"NSGA-II DESIGN Pareto — {rid}")
        ax.legend(fontsize=7)
        fig.tight_layout(); fig.savefig(FIG/f"nsga2_design_pareto_{rid}.png",dpi=180); plt.close(fig)


## 6. Structured DESIGN search — full-domain search with pre-declared local refinement

The two structured families are the symmetric four-HAPS ring and center-plus-three. The full-domain search uses the same 15° rotation step and 21 radii from 0 to the placement limit. Local refinement is DESIGN-only and follows the pre-declared rule in the configuration; TEST is never inspected.

Every completed city/family/stage is written atomically. If interruption occurs within a stage, the persistent placement-evaluation cache makes completed candidate evaluations reusable on restart.


In [ ]:
if RUN_FULL:
    def symmetric_ring(phi,r):
        th=np.mod(phi+np.arange(4,dtype=float)*(2*np.pi/4),2*np.pi)
        return th.tolist(),np.full(4,float(r)).tolist()
    def center_plus_three(phi,r):
        outer=np.mod(phi+np.arange(3,dtype=float)*(2*np.pi/3),2*np.pi)
        return np.concatenate([[0.0],outer]).tolist(),np.concatenate([[0.0],np.full(3,float(r))]).tolist()
    FAMILIES={"symmetric_ring":symmetric_ring,"center_plus_three":center_plus_three}

    theta_struct=np.radians(np.arange(0,360,float(CFG["structured"]["rotation_step_deg"])))
    broad_r=np.linspace(0,R_MAX_M,int(CFG["structured"]["broad_n_radii"]))
    A=np.arange(0,12000+1,1000,dtype=float)
    B=np.arange(0,2000+1,250,dtype=float)

    def physical_signature(thetas,rs):
        th=np.asarray(thetas,float); rr=np.asarray(rs,float)
        pts=[]
        for e,n,r in zip(rr*np.cos(th),rr*np.sin(th),rr):
            if abs(r)<1e-12: e=n=0.0
            pts.append((round(float(e),6),round(float(n),6)))
        return tuple(sorted(pts))

    def unique_candidates(family,radii,stage):
        gen=FAMILIES[family]; seen={}; rows=[]
        for phi in theta_struct:
            for r in radii:
                th,rr=gen(float(phi),float(r)); sig=physical_signature(th,rr)
                if sig not in seen:
                    seen[sig]=True
                    rows.append({"family":family,"stage":stage,"phi_rad":float(phi),
                                 "phi_deg":float(np.degrees(phi)),"r_m":float(r),
                                 "thetas":th,"rs":rr,"physical_signature":repr(sig)})
        return pd.DataFrame(rows)

    def evaluate_structured_stage(rid,family,candidates,stage):
        sdir=RAW/rid/"structured"; sdir.mkdir(parents=True,exist_ok=True)
        ck=sdir/f"{family}_{stage}.csv"
        if ck.is_file():
            out=pd.read_csv(ck)
            print(f"[SKIP] {rid} | {family} | {stage}: completed stage loaded ({len(out)})")
            return out
        rows=[]; t0=time.perf_counter()
        for i,(_,row) in enumerate(candidates.iterrows(),1):
            rec=score_placement(rid,row.thetas,row.rs,"design")
            rows.append({**row.to_dict(),"roi_id":rid,
                         "design_f1":rec["f1"],"design_f2":rec["peb95_median"]})
            if i==1 or i%20==0 or i==len(candidates):
                atomic_write_json(sdir/f"{family}_{stage}_progress.json",{
                    "roi_id":rid,"family":family,"stage":stage,"completed":i,
                    "total":len(candidates),"elapsed_s":time.perf_counter()-t0
                })
        out=pd.DataFrame(rows)
        atomic_write_df(ck,out)
        return out

    structured_design_rows=[]
    refinement_decisions=[]
    for rid in SCENES:
        for family in FAMILIES:
            # Full-domain stage always runs.
            broad=unique_candidates(family,broad_r,"broad")
            bdf=evaluate_structured_stage(rid,family,broad,"broad")
            bfront=nondominated_rows(bdf)
            smallest=float(broad_r[1])
            runA=bool(np.any(np.isclose(bfront.r_m.to_numpy(float),smallest)))
            parts=[bdf]
            runB=False

            # Local refinement is activated only by DESIGN evidence from this city/family.
            if runA:
                adf=unique_candidates(family,A,"A")
                adf=evaluate_structured_stage(rid,family,adf,"A")
                parts.append(adf)
                afront=nondominated_rows(adf)
                runB=bool(len(afront) and np.any(np.isclose(afront.r_m.to_numpy(float),1000.0)))

            if runB:
                cdf=unique_candidates(family,B,"B")
                # Stage B duplicates of Stage A are unnecessary; retain only new physical layouts.
                known=set(adf["physical_signature"].astype(str))
                cdf=cdf[~cdf["physical_signature"].astype(str).isin(known)].reset_index(drop=True)
                if len(cdf):
                    cdf=evaluate_structured_stage(rid,family,cdf,"B")
                    parts.append(cdf)

            all_df=pd.concat(parts,ignore_index=True)
            # Deduplicate the same physical layout across stages before the final DESIGN front.
            all_df=all_df.sort_values(["stage","phi_deg","r_m"]).drop_duplicates(
                subset=["physical_signature"],keep="last"
            ).reset_index(drop=True)
            final_front=nondominated_rows(all_df)
            final_front["algorithm"]=family
            final_front["seed"]="structured"
            final_front["placement_id"]=[f"{family}_{rid}_{i:03d}" for i in range(len(final_front))]
            structured_design_rows.extend(final_front.to_dict("records"))
            refinement_decisions.append({
                "roi_id":rid,"family":family,
                "broad_front_touched_smallest_positive_radius":runA,
                "stage_A_run":runA,"stage_B_run":runB,"n_final_front":len(final_front)
            })
            print(f"[OK] {rid} | {family}: front={len(final_front)} | StageA={runA} | StageB={runB}")

    structured_design_df=pd.DataFrame(structured_design_rows)
    atomic_write_df(RAW/"structured_design_fronts.csv",structured_design_df)
    atomic_write_df(RAW/"structured_refinement_decisions.csv",pd.DataFrame(refinement_decisions))


## 7. DESIGN-only immutable freeze and reporting selection

Layout selection is completed before held-out TEST evaluation. If a valid `DESIGN_FREEZE.json` already exists, the notebook **loads and verifies it rather than reselecting layouts**. Otherwise it constructs the fixed DESIGN candidate set, selects the allowed paper representatives using DESIGN only, writes all files atomically, hashes them, and writes the freeze record last.

Greedy and DE optimize solvability only and therefore receive only recovery-selected headline representatives. DESIGN max-R/min-PEB extremes are reported only for NSGA-II and the structured multi-objective candidate sets.


In [ ]:
if RUN_FULL:
    freeze_path=RAW/"DESIGN_FREEZE.json"
    frozen_csv=RAW/"frozen_design_layouts.csv"
    paper_csv=RAW/"paper_representatives_design_selected.csv"
    seed_csv=RAW/"seed_representatives_design_selected.csv"

    if freeze_path.is_file():
        meta=json.loads(freeze_path.read_text(encoding="utf-8"))
        for p,k in [(frozen_csv,"frozen_layouts_sha256"),(paper_csv,"paper_representatives_sha256"),(seed_csv,"seed_representatives_sha256")]:
            if not p.is_file() or sha256_file(p)!=meta[k]:
                raise RuntimeError(f"Frozen DESIGN artifact missing or hash-mismatched: {p}")
        if meta.get("protocol_version")!=PROTOCOL_VERSION or meta.get("input_fingerprint_sha256")!=INPUT_FINGERPRINT_SHA256:
            raise RuntimeError("DESIGN freeze belongs to a different protocol/input fingerprint")
        frozen_df=pd.read_csv(frozen_csv)
        paper_reps=pd.read_csv(paper_csv)
        seed_reps=pd.read_csv(seed_csv)
        print(f"[RESUME] immutable DESIGN freeze loaded | layouts={len(frozen_df)} | reps={len(paper_reps)}")
    else:
        frozen_rows=[]
        # Greedy: one deterministic four-HAPS recovery solution.
        for rid,g in GREEDY.items():
            rec=score_placement(rid,g["thetas"],g["rs"],"design")
            frozen_rows.append({
                "roi_id":rid,"algorithm":"greedy_add","seed":"deterministic","placement_id":"greedy4",
                "design_f1":rec["f1"],"design_f2":rec["peb95_median"],
                **{f"theta_{j+1}":g["thetas"][j] for j in range(4)},
                **{f"r_{j+1}_m":g["rs"][j] for j in range(4)}
            })
        # DE: one DESIGN-R solution per seed.
        for rid,rrr in DE_RESULTS.items():
            for lbl,d in rrr.items():
                th,rad=decode_x(d["meta"]["best_x"])
                frozen_rows.append({
                    "roi_id":rid,"algorithm":"differential_evolution","seed":lbl,
                    "placement_id":f"de_best_{lbl}",
                    "design_f1":d["meta"]["best_f1"],"design_f2":d["meta"]["best_f2"],
                    **{f"theta_{j+1}":th[j] for j in range(4)},
                    **{f"r_{j+1}_m":rad[j] for j in range(4)}
                })
        # NSGA-II DESIGN fronts.
        for rid,rrr in NSGA_RESULTS.items():
            for lbl,d in rrr.items():
                for n,(_,row) in enumerate(d["front"].iterrows()):
                    frozen_rows.append({
                        "roi_id":rid,"algorithm":"nsga2","seed":lbl,
                        "placement_id":f"nsga2_{lbl}_f{n:03d}",
                        "design_f1":float(row["f1"]),"design_f2":float(row["f2"]),
                        **{f"theta_{j+1}":float(row[f"x_theta{j+1}"]) for j in range(4)},
                        **{f"r_{j+1}_m":float(row[f"x_r{j+1}"]) for j in range(4)}
                    })
        # Structured DESIGN fronts.
        for _,row in structured_design_df.iterrows():
            th=row["thetas"] if isinstance(row["thetas"],(list,tuple,np.ndarray)) else json.loads(str(row["thetas"]).replace("'",'"'))
            rr=row["rs"] if isinstance(row["rs"],(list,tuple,np.ndarray)) else json.loads(str(row["rs"]).replace("'",'"'))
            frozen_rows.append({
                "roi_id":row["roi_id"],"algorithm":row["algorithm"],"seed":"structured",
                "placement_id":row["placement_id"],
                "design_f1":float(row["design_f1"]),"design_f2":float(row["design_f2"]),
                **{f"theta_{j+1}":float(th[j]) for j in range(4)},
                **{f"r_{j+1}_m":float(rr[j]) for j in range(4)}
            })

        frozen_df=pd.DataFrame(frozen_rows)
        paper_reps=select_paper_representatives(frozen_df)
        seed_reps=select_seed_representatives(frozen_df)

        atomic_write_df(frozen_csv,frozen_df)
        atomic_write_df(paper_csv,paper_reps)
        atomic_write_df(seed_csv,seed_reps)
        freeze_meta={
            "protocol_version":PROTOCOL_VERSION,
            "frozen_at_utc":datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "design_epochs":[str(x) for x in DESIGN_EPOCHS],
            "test_epochs":[str(x) for x in TEST_EPOCHS],
            "input_fingerprint_sha256":INPUT_FINGERPRINT_SHA256,
            "frozen_layouts_sha256":sha256_file(frozen_csv),
            "paper_representatives_sha256":sha256_file(paper_csv),
            "seed_representatives_sha256":sha256_file(seed_csv),
            "selection_rule":"all paper/seed representatives selected only from DESIGN metrics before TEST",
            "single_objective_reporting":"Greedy/DE have recovery-selected representatives only; min-PEB extrema are reserved for multi-objective candidate sets",
            "test_used_for_selection":False
        }
        # Record DESIGN selection after saving all selected layouts and representatives.
        atomic_write_json(freeze_path,freeze_meta)
        print(f"[OK] DESIGN freeze complete | layouts={len(frozen_df)} | reps={len(paper_reps)}")

    display(paper_reps[["roi_id","algorithm","placement_id","selection_rule","design_f1","design_f2"]])


## 8. Held-out TEST evaluation

The paper and seed-level representatives are selected using DESIGN and recorded before held-out TEST evaluation.

TEST data are used only to evaluate the selected layouts. They do not influence seed selection, radii, layout choices, methods, thresholds, or Pareto-point selection.


In [ ]:
if RUN_FULL:
    # Form the TEST evaluation set from DESIGN-selected headline and seed representatives.
    eval_reps=pd.concat([paper_reps,seed_reps],ignore_index=True)
    eval_reps=eval_reps.drop_duplicates(subset=["roi_id","algorithm","placement_id"]).reset_index(drop=True)

    test_rows=[]; common_rows=[]
    for _,row in eval_reps.iterrows():
        rid=row["roi_id"]; pid=row["placement_id"]
        edir=RAW/rid/"test_checkpoint"; edir.mkdir(parents=True,exist_ok=True)
        summary_path=edir/f"{pid}.json"; detail_path=edir/f"{pid}_detail.csv"
        if summary_path.is_file() and detail_path.is_file():
            outrow=json.loads(summary_path.read_text(encoding="utf-8"))
            det=pd.read_csv(detail_path)
            print(f"[SKIP] TEST {rid} {pid}: checkpoint loaded")
        else:
            th=[float(row[f"theta_{j}"]) for j in range(1,5)]
            rr=[float(row[f"r_{j}_m"]) for j in range(1,5)]
            rec=score_placement(rid,th,rr,"test",detail=True)
            outrow={
                "roi_id":rid,"algorithm":row["algorithm"],"seed":row["seed"],
                "placement_id":pid,
                "test_R":rec["f1"],"test_conditional_median_PEB95_m":rec["peb95_median"],
                "test_PEB95_p95_m":rec["peb95_p95"],"n_gnss_unsolved_test":rec["n_unpos"],
                "n_made_solvable_test":rec["n_p2"]
            }
            det=rec["_detail"].copy()
            atomic_write_df(detail_path,det)
            atomic_write_json(summary_path,outrow)
        test_rows.append(outrow)
        if len(det):
            ec=common_population_ecdf(pd.to_numeric(det["peb95_m"],errors="coerce").to_numpy(float))
            ec["roi_id"]=rid; ec["algorithm"]=row["algorithm"]; ec["seed"]=row["seed"]
            ec["placement_id"]=pid
            common_rows.append(ec)

    # Evaluate the fixed 300 m reference layout using the GPS/Galileo clock-state model.
    ref_rows=[]
    for rid in SCENES:
        edir=RAW/rid/"test_checkpoint"; edir.mkdir(parents=True,exist_ok=True)
        sp=edir/"preserved_c4_reference.json"; dp=edir/"preserved_c4_reference_detail.csv"
        if sp.is_file() and dp.is_file():
            refrow=json.loads(sp.read_text(encoding="utf-8")); det=pd.read_csv(dp)
            print(f"[SKIP] reference TEST {rid}: checkpoint loaded")
        else:
            drec=score_fixed_reference(rid,"design",detail=False)
            trec=score_fixed_reference(rid,"test",detail=True)
            refrow={
                "roi_id":rid,"algorithm":"preserved_c4_reference","seed":"deterministic",
                "placement_id":"c4_300m_24h_phase",
                "design_f1":drec["f1"],"design_f2":drec["peb95_median"],
                "test_R":trec["f1"],"test_conditional_median_PEB95_m":trec["peb95_median"],
                "test_PEB95_p95_m":trec["peb95_p95"],"n_gnss_unsolved_test":trec["n_unpos"],
                "n_made_solvable_test":trec["n_p2"],"selection_rule":"reference"
            }
            det=trec["_detail"].copy()
            atomic_write_df(dp,det); atomic_write_json(sp,refrow)
        ref_rows.append(refrow)
        if len(det):
            ec=common_population_ecdf(pd.to_numeric(det["peb95_m"],errors="coerce").to_numpy(float))
            ec["roi_id"]=rid; ec["algorithm"]="preserved_c4_reference"; ec["seed"]="deterministic"
            ec["placement_id"]="c4_300m_24h_phase"
            common_rows.append(ec)

    test_df=pd.DataFrame(test_rows)
    reference_df=pd.DataFrame(ref_rows)
    common_ecdf_df=pd.concat(common_rows,ignore_index=True) if common_rows else pd.DataFrame()
    atomic_write_df(RAW/"test_results_design_selected.csv",test_df)
    atomic_write_df(RAW/"fixed_reference_results.csv",reference_df)
    atomic_write_df(RAW/"common_population_peb_ecdf.csv",common_ecdf_df)

    # Join TEST results with representatives selected on DESIGN.
    headline=paper_reps.merge(
        test_df,on=["roi_id","algorithm","seed","placement_id"],how="left",validate="many_to_one"
    )
    headline=pd.concat([headline,reference_df],ignore_index=True,sort=False)
    atomic_write_df(RAW/"camera_ready_headline_results.csv",headline)

    # Stochastic variability: each seed representative was fixed from DESIGN.
    seed_eval=seed_reps.merge(
        test_df,on=["roi_id","algorithm","seed","placement_id"],how="left",validate="many_to_one"
    )
    seed_stats=(seed_eval.groupby(["roi_id","algorithm","selection_rule"])
                .agg(n_seeds=("seed","nunique"),
                     test_R_mean=("test_R","mean"),test_R_std=("test_R","std"),
                     test_PEB_mean=("test_conditional_median_PEB95_m","mean"),
                     test_PEB_std=("test_conditional_median_PEB95_m","std"))
                .reset_index())
    atomic_write_df(RAW/"seed_test_statistics.csv",seed_stats)

    display(headline[["roi_id","algorithm","selection_rule","design_f1","design_f2",
                      "test_R","test_conditional_median_PEB95_m"]])
    display(seed_stats)


## 9. Paper evidence and provenance

The Pareto plots are DESIGN evidence. TEST is reported only for the already-fixed representatives. The fixed-denominator PEB curve gives a like-for-like quality comparison across layouts by counting every originally GNSS-unsolved TEST receiver-time case in the denominator; cases not made solvable remain at infinite PEB and never enter a finite threshold.

The final provenance record includes the DESIGN-selection hashes, full input fingerprint, protocol version, and explicit remaining limitations.


In [ ]:

if RUN_FULL:
    import matplotlib.pyplot as plt

    # DESIGN Pareto fronts (no TEST-based selection).
    for rid in SCENES:
        fig,ax=plt.subplots(figsize=(6.4,4.6))
        sub=frozen_df[frozen_df.roi_id==rid]
        for algo,g in sub.groupby("algorithm"):
            finite=g[np.isfinite(pd.to_numeric(g.design_f2,errors="coerce"))]
            if len(finite):
                ax.scatter(finite.design_f2,100*finite.design_f1,s=18,label=algo,alpha=.7)
        ax.set_xlabel("DESIGN conditional median PEB95,h [m]")
        ax.set_ylabel("DESIGN solvable fraction R [%]")
        ax.set_title(rid.replace("_"," "))
        ax.grid(alpha=.25); ax.legend(fontsize=7)
        fig.tight_layout(); fig.savefig(FIG/f"design_pareto_{rid}.png",dpi=300); plt.close(fig)

    # Common-population TEST ECDF.
    for rid in SCENES:
        fig,ax=plt.subplots(figsize=(6.4,4.6))
        sub=common_ecdf_df[common_ecdf_df.roi_id==rid]
        for (algo,pid),g in sub.groupby(["algorithm","placement_id"]):
            ax.step(g.peb95_threshold_m,100*g.fraction_all_gnss_unsolved,where="post",
                    label=f"{algo}:{pid}",alpha=.8)
        ax.set_xscale("log")
        ax.set_xlabel("PEB95,h threshold [m]")
        ax.set_ylabel("All GNSS-unsolved TEST cases solved within threshold [%]")
        ax.set_title(rid.replace("_"," "))
        ax.grid(alpha=.25); ax.legend(fontsize=6)
        fig.tight_layout(); fig.savefig(FIG/f"common_population_peb_{rid}.png",dpi=300); plt.close(fig)

    source_hashes={}
    for rel in CFG.get("source_hashes",{}):
        p=PROJECT_ROOT/rel
        if p.is_file(): source_hashes[rel]=sha256_file(p)

    provenance={
        "protocol_version":PROTOCOL_VERSION,
        "experiment_id":CFG["experiment_id"],
        "run_id":RUN_ID,
        "completed_utc":datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "cities":[r["id"] for r in ROIS],
        "input_fingerprint_sha256":INPUT_FINGERPRINT_SHA256,
        "design_freeze_sha256":sha256_file(RAW/"frozen_design_layouts.csv"),
        "paper_representatives_sha256":sha256_file(RAW/"paper_representatives_design_selected.csv"),
        "test_used_for_selection":False,
        "source_hashes":source_hashes,
        "limitations":[
            "geometric LOS only; no multipath, diffraction, atmospheric or detailed receiver error model",
            "5 m HAPS pseudorange uncertainty is retained from the accepted experiment",
            "same two-hour day/window retained to isolate the critical camera-ready corrections",
            "city building completeness is limited by the frozen OSM snapshot and reported explicitly"
        ]
    }
    atomic_write_json(RUN_DIR/"PROVENANCE.json",provenance)

    required=[
        RAW/"urban_geometry_summary.csv",
        RAW/"design_population_summary.csv",
        RAW/"frozen_design_layouts.csv",
        RAW/"paper_representatives_design_selected.csv",
        RAW/"seed_representatives_design_selected.csv",
        RAW/"DESIGN_FREEZE.json",
        RAW/"test_results_design_selected.csv",
        RAW/"fixed_reference_results.csv",
        RAW/"camera_ready_headline_results.csv",
        RAW/"seed_test_statistics.csv",
        RAW/"common_population_peb_ecdf.csv",
        RUN_DIR/"PROVENANCE.json"
    ]
    missing=[str(x) for x in required if not x.is_file()]
    if missing:
        raise RuntimeError("Missing required outputs:\n"+"\n".join(missing))
    print("[SUMMARY] camera-ready scientific run COMPLETE")
else:
    smoke_report={
        "status":"passed",
        "timestamp_utc":datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "tests":[
            "ENU azimuth convention",
            "dynamic 4/5-state GPS-Galileo clock model",
            "fixed-denominator common-population ECDF",
            "DESIGN-only representative selection",
            "single-objective vs multi-objective reporting rule",
            "durable JSONL cache recovery",
            "atomic CSV/JSON checkpoints"
        ]
    }
    atomic_write_json(RUN_DIR/"SMOKE_TEST_REPORT.json",smoke_report)
    print("[SUMMARY] notebook end-to-end SMOKE TEST COMPLETE")
